# CNV data access

This notebook documents the CNV (copy number variant) data access and plotting methods on the `Ag3` class: `coverage_calls_analysis_ids`, `cnv_hmm`, `cnv_coverage_calls`, `cnv_discordant_read_calls`, `plot_cnv_hmm_coverage`, `plot_cnv_hmm_heatmap` and `gene_cnv`. `Ag3` (*Anopheles gambiae* complex) provides CNV data from three independent calling pipelines: a windowed read-depth HMM (`cnv_hmm`), discrete coverage-based calls against a fixed set of known CNV alleles (`cnv_coverage_calls`), and discordant-read-pair evidence for breakpoints (`cnv_discordant_read_calls`).

**Diagram opportunity:** a flowchart showing how the three CNV calling pipelines (HMM read-depth, coverage calls against known alleles, discordant read-pair breakpoint calls) relate to each other and to the underlying aligned reads, and where each one's output (`cnv_hmm`, `cnv_coverage_calls`, `cnv_discordant_read_calls`) fits.

In [1]:
import malariagen_data

ag3 = malariagen_data.Ag3(
    "simplecache::gs://vo_agam_release_master_us_central1",
    simplecache=dict(cache_storage="../../gcs_cache"),
    results_cache="../../results_cache",
)
ag3

/opt/homebrew/Caskroom/miniconda/base/envs/malariagen2/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


<MalariaGEN Ag3 API client>
Storage URL                           : simplecache::gs://vo_agam_release_master_us_central1
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
Results cache                         : /Users/katie.barr/malariagen-data-python/results_cache
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.8.0.post13+b769b728
Client location                       : England, United Kingdom
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
---
Please note that data are subject to terms of use,
for more information see the Vector Observatory website https://www.malariagen.net/vobs/
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.8.0.post13+b769b728/Ag3.html

## `coverage_calls_analysis_ids`

A read-only property (no parameters) that returns the identifiers of the coverage-calls analyses available for this data resource. These identifier strings are the valid values for the `analysis` parameter of `cnv_coverage_calls()` (and other methods that accept a coverage-calls analysis). For `Ag3` this returns `("gamb_colu", "arab")`, reflecting that coverage-based CNV alleles were called separately for the *An. gambiae*/*An. coluzzii* complex and for *An. arabiensis*.

In [2]:
ag3.coverage_calls_analysis_ids

('gamb_colu', 'arab')

## `cnv_hmm`

Accesses genome-wide CNV HMM (hidden Markov model) calls: for each sample, a modelled copy number and normalised/raw coverage value is given for consecutive genomic windows across the requested region(s). Parameters:

- `region` (required): one or more genome regions (contig, region string, or gene/transcript ID) to restrict the windows returned. Passing a list concatenates data from multiple regions.
- `sample_sets`: which sample set(s) or release(s) to draw samples from; defaults to all available sample sets if omitted.
- `sample_query` / `sample_query_options`: a pandas query string (and options passed to `DataFrame.eval`) evaluated against sample metadata, to restrict to a subset of samples, e.g. by taxon or country.
- `max_coverage_variance`: samples whose coverage variance exceeds this threshold are dropped (default `0.2`); set to `None` to keep all samples regardless of coverage-variance quality.
- `inline_array` / `chunks`: low-level dask/zarr loading controls (whether small arrays are inlined into the dask graph, and how the arrays are chunked); rarely need changing from defaults.

In [3]:
ds_hmm = ag3.cnv_hmm(
    region="3L:15,000,000-16,000,000",
    sample_sets="AG1000G-BF-A",
    sample_query="taxon == 'coluzzii'",
    max_coverage_variance=0.2,
)
ds_hmm

Access CNV HMM data: ⠋ (0:00:00.00)

Access CNV HMM data: ⠙ (0:00:00.09)

Load sample metadata: ⠋ (0:00:00.00)

<xarray.Dataset> Size: 2MB
Dimensions:                   (variants: 3335, samples: 80)
Coordinates:
    variant_position          (variants) int32 13kB dask.array<chunksize=(3335,), meta=np.ndarray>
    variant_end               (variants) int32 13kB dask.array<chunksize=(3335,), meta=np.ndarray>
    variant_contig            (variants) uint8 3kB dask.array<chunksize=(3335,), meta=np.ndarray>
    sample_id                 (samples) object 640B dask.array<chunksize=(80,), meta=np.ndarray>
Dimensions without coordinates: variants, samples
Data variables:
    call_CN                   (variants, samples) int8 267kB dask.array<chunksize=(3335, 41), meta=np.ndarray>
    call_RawCov               (variants, samples) int32 1MB dask.array<chunksize=(3335, 41), meta=np.ndarray>
    call_NormCov              (variants, samples) float32 1MB dask.array<chunksize=(3335, 41), meta=np.ndarray>
    sample_coverage_variance  (samples) float32 320B dask.array<chunksize=(80,), meta=np.ndarray>
    sample_is_high_variance   (samples) bool 80B dask.array<chunksize=(80,), meta=np.ndarray>
Attributes:
    contigs:  ('2R', '2L', '3R', '3L', 'X')

## `cnv_coverage_calls`

Accesses discrete CNV calls made against a curated set of known CNV alleles, based on normalised coverage. Unlike `cnv_hmm`, calling is performed independently per sample set, so this method takes a single `sample_set` (not `sample_sets`) and the alleles returned can differ between sample sets. Parameters:

- `region` (required): genome region(s) to restrict the CNV alleles returned.
- `sample_set` (required): a single sample set identifier to call within (confirmed via `ag3.sample_sets()`).
- `analysis` (required): which coverage-calls analysis to use — one of the values from `coverage_calls_analysis_ids`, e.g. `"gamb_colu"`.
- `inline_array` / `chunks`: low-level dask/zarr loading controls, as above.

In [4]:
ds_cov_calls = ag3.cnv_coverage_calls(
    region="3L:15,000,000-16,000,000",
    sample_set="AG1000G-BF-A",
    analysis="gamb_colu",
)
ds_cov_calls

<xarray.Dataset> Size: 84kB
Dimensions:              (variants: 403, samples: 178)
Coordinates:
    variant_position     (variants) int32 2kB dask.array<chunksize=(403,), meta=np.ndarray>
    variant_end          (variants) int32 2kB dask.array<chunksize=(403,), meta=np.ndarray>
    variant_contig       (variants) uint8 403B dask.array<chunksize=(403,), meta=np.ndarray>
    variant_id           (variants) object 3kB dask.array<chunksize=(403,), meta=np.ndarray>
    sample_id            (samples) object 1kB dask.array<chunksize=(178,), meta=np.ndarray>
Dimensions without coordinates: variants, samples
Data variables:
    variant_CIPOS        (variants) int32 2kB dask.array<chunksize=(403,), meta=np.ndarray>
    variant_CIEND        (variants) int32 2kB dask.array<chunksize=(403,), meta=np.ndarray>
    variant_filter_pass  (variants) bool 403B dask.array<chunksize=(403,), meta=np.ndarray>
    call_genotype        (variants, samples) int8 72kB dask.array<chunksize=(403, 64), meta=np.ndarray>
Attributes:
    contigs:  ('2R', '2L', '3R', '3L', 'X')

## `cnv_discordant_read_calls`

Accesses CNV calls derived from discordant read-pair evidence (read pairs whose mapping distance/orientation implies a structural variant breakpoint). Because some of these CNV alleles have unknown/imprecise start or end coordinates, this method takes whole `contigs` rather than a `region` with start/end coordinates. Parameters:

- `contigs` (required, keyword or positional): a contig name or list of contig names to return discordant-read CNV calls for.
- `contig`: deprecated alias for a single contig; using it raises a `DeprecationWarning` and is provided only for backwards compatibility.
- `sample_sets`: which sample set(s)/release(s) to draw samples from.
- `sample_query` / `sample_query_options`: restrict samples via a pandas query against sample metadata.
- `inline_array` / `chunks`: low-level dask/zarr loading controls.

Not every contig has discordant-read CNV data for every sample set (calling is only run where relevant); here we use `"2R"`, which includes the well-studied Cyp6aa/p resistance locus.

In [5]:
ds_drc = ag3.cnv_discordant_read_calls(
    contigs="2R",
    sample_sets="AG1000G-BF-A",
    sample_query="taxon == 'coluzzii'",
)
ds_drc

<xarray.Dataset> Size: 6kB
Dimensions:                        (variants: 47, samples: 82)
Coordinates:
    variant_position               (variants) int32 188B dask.array<chunksize=(47,), meta=np.ndarray>
    variant_end                    (variants) int32 188B dask.array<chunksize=(47,), meta=np.ndarray>
    variant_id                     (variants) object 376B dask.array<chunksize=(47,), meta=np.ndarray>
    variant_contig                 (variants) uint8 47B dask.array<chunksize=(47,), meta=np.ndarray>
    sample_id                      (samples) object 656B dask.array<chunksize=(82,), meta=np.ndarray>
Dimensions without coordinates: variants, samples
Data variables:
    variant_Region                 (variants) object 376B dask.array<chunksize=(47,), meta=np.ndarray>
    variant_StartBreakpointMethod  (variants) int32 188B dask.array<chunksize=(47,), meta=np.ndarray>
    variant_EndBreakpointMethod    (variants) int32 188B dask.array<chunksize=(47,), meta=np.ndarray>
    call_genotype                  (variants, samples) int8 4kB dask.array<chunksize=(47, 60), meta=np.ndarray>
    sample_coverage_variance       (samples) float32 328B dask.array<chunksize=(82,), meta=np.ndarray>
    sample_is_high_variance        (samples) bool 82B dask.array<chunksize=(82,), meta=np.ndarray>
Attributes:
    contigs:  ('2R', '2L', '3R', '3L', 'X')

## `plot_cnv_hmm_coverage`

Plots CNV HMM data (normalised coverage points and the fitted HMM copy-number state) for a **single sample** across a region, using bokeh, together with a genes track underneath so CNV breakpoints can be related to gene positions. This combines `plot_cnv_hmm_coverage_track` (the coverage/HMM track) with `plot_genes` (the genes track). Parameters:

- `sample` (required): sample identifier to plot.
- `region` (required): genome region to plot.
- `sample_set`: sample set the sample belongs to; if omitted it is looked up automatically from sample metadata.
- `y_max`: upper limit of the copy-number y-axis, or `"auto"` (default) to size it to the data (max observed CN + 2).
- `sizing_mode`, `width`: overall bokeh figure sizing/layout controls.
- `track_height`: height in pixels of the coverage/HMM track.
- `genes_height`: height in pixels of the genes track underneath.
- `circle_kwargs` / `line_kwargs`: dicts of extra bokeh styling passed to the coverage-point scatter and the HMM-state line respectively (e.g. colour, size).
- `show`: if `True` (default) immediately display the figure; set `False` to only return it.
- `output_backend`: bokeh rendering backend (e.g. `"canvas"` vs `"svg"`).
- `gene_labels` / `gene_labelset`: control which gene labels are drawn on the genes track.

The example below uses a sample known to carry a large CNV in the Cyp6aa/p region on 2R, so the HMM copy-number step is clearly visible.

In [6]:
ag3.plot_cnv_hmm_coverage(
    sample="AY0072-C",
    region="2R:28,460,000-28,580,000",
)

Load sample metadata: ⠋ (0:00:00.00)

Load sample metadata: ⠙ (0:00:00.10)

Load sample metadata: ⠹ (0:00:00.19)

Load sample metadata: ⠸ (0:00:00.28)

Load sample metadata: ⠼ (0:00:00.37)

...

Load genome features: ⠋ (0:00:00.00)

Load genome features: ⠙ (0:00:00.08)

Load genome features: ⠹ (0:00:00.18)

Load genome features: ⠸ (0:00:00.36)

GridPlot(id='p1113', ...)

## `plot_cnv_hmm_heatmap`

Plots CNV HMM copy-number state for **multiple samples** at once, as a heatmap (one row per sample, coloured by copy-number state), together with a genes track underneath. This combines `plot_cnv_hmm_heatmap_track` with `plot_genes`. Parameters:

- `region` (required): genome region to plot.
- `sample_sets`: which sample set(s)/release(s) to include.
- `sample_query` / `sample_query_options`: restrict to a sample subset via a pandas query.
- `max_coverage_variance`: drop samples with coverage variance above this threshold (default `0.2`); `None` keeps all samples.
- `sizing_mode`, `width`: overall bokeh figure sizing/layout controls.
- `row_height`: pixel height allotted per sample row (controls total heatmap height together with the number of samples).
- `track_height`: explicit total height of the heatmap track (overrides the `row_height`-based calculation if given).
- `palette`: list of colours used for the copy-number colour scale; defaults to a grey/purple/orange diverging scale centred on the normal (2-copy) state.
- `genes_height`: height of the genes track.
- `show`: if `True` (default) immediately display the figure.
- `gene_labels` / `gene_labelset`: control gene labels on the genes track.

**Diagram opportunity:** a labelled illustration of the heatmap colour scale, showing what each copy-number colour band means (e.g. -1 = unknown, 2 = normal diploid, 4+ = high amplification), since the colour mapping is fixed (-1.5 to 4.5) and not otherwise obvious from the plot alone.

In [7]:
ag3.plot_cnv_hmm_heatmap(
    sample_sets="AG1000G-BF-A",
    sample_query="taxon == 'coluzzii'",
    region="2R:28,460,000-28,580,000",
)

Access CNV HMM data: ⠋ (0:00:00.00)

Load genome features: ⠋ (0:00:00.00)

GridPlot(id='p1240', ...)

## `gene_cnv`

Computes the **modal copy number per gene** for each sample, by aggregating the windowed `cnv_hmm` calls that overlap each gene in the region and taking the most frequent (modal) copy-number value. This is the building block used by `gene_cnv_frequencies` and `gene_cnv_frequencies_advanced`. Parameters:

- `region` (required): genome region(s), or a gene/transcript ID, defining which genes to compute modal copy number for.
- `sample_sets`: which sample set(s)/release(s) to draw samples from.
- `sample_query` / `sample_query_options`: restrict to a sample subset via a pandas query.
- `max_coverage_variance`: drop samples with coverage variance above this threshold before computing modal CN (default `0.2`); `None` keeps all samples.
- `chunks` / `inline_array`: low-level dask/zarr loading controls.

The output dataset has a `genes` dimension (one entry per gene in the region) and a `samples` dimension, with `CN_mode` giving the modal copy number and `CN_mode_count` giving how many HMM windows supported that mode.

**Diagram opportunity:** a genomic diagram showing a gene's exon/intron structure with the overlapping HMM windows underneath, and an arrow showing how the per-window copy-number calls are aggregated into a single modal copy number for the gene — this is the concept underlying "gene CNV frequency" used throughout this notebook and notebook 3.

In [8]:
ds_gene_cnv = ag3.gene_cnv(
    region="AGAP004707",
    sample_sets="AG1000G-BF-A",
    sample_query="taxon == 'coluzzii'",
)
ds_gene_cnv

Load genome features: ⠋ (0:00:00.00)

Load genome features: ⠙ (0:00:00.09)

Load genome features: ⠹ (0:00:00.18)

Load genome features: ⠸ (0:00:00.34)

Load genome features: ⠋ (0:00:00.00)

...

Access CNV HMM data: ⠧ (0:00:01.46)

Access CNV HMM data: ⠇ (0:00:01.55)

Load CNV HMM data:   0%|          | 0/23 [00:00<?, ?it/s]

Compute modal gene copy number:   0%|          | 0/1 [00:00<?, ?it/s]

<xarray.Dataset> Size: 2kB
Dimensions:                   (genes: 1, samples: 80)
Coordinates:
    gene_id                   (genes) object 8B 'AGAP004707'
    sample_id                 (samples) object 640B 'AB0087-C' ... 'AB0282-Cx'
Dimensions without coordinates: genes, samples
Data variables:
    gene_contig               (genes) object 8B '2L'
    gene_start                (genes) int64 8B 2358158
    gene_end                  (genes) int64 8B 2431617
    gene_windows              (genes) int64 8B 246
    gene_name                 (genes) object 8B 'para'
    gene_strand               (genes) object 8B '+'
    gene_description          (genes) object 8B 'voltage-gated sodium channel...
    CN_mode                   (genes, samples) int8 80B 2 2 2 2 2 ... 2 2 2 2 2
    CN_mode_count             (genes, samples) int64 640B 199 191 ... 181 199
    sample_coverage_variance  (samples) float32 320B 0.07554 0.06393 ... 0.192
    sample_is_high_variance   (samples) bool 80B False False ... False False